## Phase 5 — Train ML Models

In [ ]:

from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import time

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_models():
    """Always returns FRESH, unfitted model instances."""
    return {
        'kNN':           KNeighborsClassifier(n_neighbors=8),
        'MLP':           MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42),
        'Random Forest': RandomForestClassifier(max_depth=16, n_estimators=50, random_state=42),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'Grad. Boost':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    }

# ── Scale features for MLP
scaler  = StandardScaler()
scaler2 = StandardScaler()
X_train_bin_scaled = scaler.fit_transform(X_train_bin)
X_test_bin_scaled  = scaler.transform(X_test_bin)
X_train_mul_scaled = scaler2.fit_transform(X_train_mul)
X_test_mul_scaled  = scaler2.transform(X_test_mul)

def train_models(X_train, y_train,
                 X_train_scaled, case_name):
    print(f"\n{'='*50}")
    print(f"  {case_name}")
    print(f"{'='*50}")
    trained = {}
    for name, model in make_models().items():        # fresh models every time!
        start = time.time()
        X_tr = X_train_scaled if name == 'MLP' else X_train
        cv_scores = cross_val_score(model, X_tr, y_train, cv=cv, scoring='accuracy')
        model.fit(X_tr, y_train)
        elapsed = time.time() - start
        trained[name] = model
        print(f"  {name:<20} CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  ({elapsed:.1f}s)")
    return trained

print("Training started — this may take a few minutes...")

trained_bin = train_models(X_train_bin, y_train_bin,
                            X_train_bin_scaled,
                            "Case 1 — Binary (Benign vs Darknet)")

trained_mul = train_models(X_train_mul, y_train_mul,
                            X_train_mul_scaled,
                            "Case 2 — Multiclass (Tor/Non-Tor/VPN/Non-VPN)")

print("\nAll models trained successfully!")